<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎤 MOSS-TTS v1.5 - Standalone Foundation Model</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Kaggle T4 x2 GPU Edition - Created by <strong>AIQUEST Academy</strong></h3>
  <p style='color: #ddd; margin: 0; text-align: center;'>High-Fidelity 48 kHz Stereo TTS and Zero-Shot Voice Cloning</p>
</div>

<div align="center">
  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Kaggle-T4%20GPU%20x2-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white" />
  <br>
  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
</div>

### Setup Instructions
1. **Settings -> Accelerator -> GPU T4 x2** (MUST select dual T4 GPUs)
2. Run all cells **top to bottom**
3. Use the public Gradio link to open the web interface

---

## ⚙️ Cell 1 - Environment and GPU Memory Optimization

In [ ]:
# Cell 1: Check GPU and Optimize Environment Memory — Kaggle T4 x2 Robust Version
import os
import gc
import sys
import torch

# psutil is optional on Kaggle - provide fallback
try:
    import psutil
    HAS_PSUTIL = True
except ImportError:
    HAS_PSUTIL = False
    print("psutil not found - will skip RAM detailed check (install via pip if needed)")
    # auto-install psutil quietly
    import subprocess
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "psutil"])
        import psutil
        HAS_PSUTIL = True
        print("psutil installed successfully")
    except Exception as e:
        print(f"Could not install psutil: {e}")

print("=== Kaggle T4 Environment Setup ===")
print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
if HAS_PSUTIL:
    try:
        vm = psutil.virtual_memory()
        print(f"RAM: {vm.total / 1024**3:.1f} GB total, {vm.available / 1024**3:.1f} GB available")
    except Exception as e:
        print(f"RAM check skipped: {e}")
else:
    print("RAM check skipped (psutil unavailable)")

# Important: Set memory allocator config BEFORE heavy CUDA allocations.
# On Kaggle the kernel already loaded torch, so we set it now and warn if restart needed.
# This setting helps prevent fragmentation on 16GB T4s.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.6"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Also set HF cache early so even processor download uses /kaggle/tmp (large scratch)
# /kaggle/tmp is ~70GB vs /root 20GB overlay - prevents "No space left on device"
for _cand in ["/kaggle/tmp/hf_cache", "/tmp/hf_cache"]:
    try:
        os.makedirs(_cand, exist_ok=True)
        break
    except: pass
# Prefer /kaggle/tmp if exists (Kaggle), else /tmp (Colab/local)
hf_cache_dir = "/kaggle/tmp/hf_cache" if os.path.isdir("/kaggle/tmp") else "/tmp/hf_cache"
os.environ["HF_HOME"] = hf_cache_dir
os.environ["HF_HUB_CACHE"] = hf_cache_dir
os.environ["TRANSFORMERS_CACHE"] = hf_cache_dir
print(f"HF cache set to: {hf_cache_dir}")

gc.collect()
if torch.cuda.is_available():
    try:
        torch.cuda.empty_cache()
    except: pass

if torch.cuda.is_available():
    device_count = torch.cuda.device_count()
    print(f"Number of GPUs Available: {device_count}")
    for i in range(device_count):
        try:
            device_name = torch.cuda.get_device_name(i)
            total_memory = torch.cuda.get_device_properties(i).total_memory / 1e9
            cap = torch.cuda.get_device_capability(i)
            print(f"  GPU {i}: {device_name} ({total_memory:.2f} GB VRAM, compute capability {cap[0]}.{cap[1]})")
        except Exception as e:
            print(f"  GPU {i}: error querying: {e}")
    if device_count < 2:
        print("⚠️  Only 1 GPU detected. Go to Settings -> Accelerator -> GPU T4 x2 to enable dual GPUs.")
        print("   Notebook will still run on 1 GPU but is slower and more likely to OOM - reduce max_new_tokens if needed.")
else:
    print("⚠️  No GPU detected. Go to Settings -> Accelerator -> GPU T4 x2")
    print("   On Kaggle: Notebook Settings (right panel) -> Accelerator -> GPU T4 x2 + Internet ON")

# Attention backend flags for Turing GPUs (T4 = compute capability 7.5).
# T4 has NO bfloat16 and NO FlashAttention-2 (both need Ampere 8.0+), so:
#   - disable the broken cuDNN SDPA backend (recommended by OpenMOSS)
#   - disable flash_sdp (unsupported on T4)
#   - keep mem_efficient + math SDPA as fp16-compatible fallbacks
# Wrap in try/except because API changed in torch 2.4+ (enable_* moved)
try:
    torch.backends.cuda.enable_cudnn_sdp(False)
    print("  cuDNN SDPA disabled (T4 fix)")
except AttributeError:
    try:
        torch.backends.cuda.matmul.allow_tf32 = True
    except: pass
    print("  cuDNN SDPA flag not available (torch version difference) - skipping")
except Exception as e:
    print(f"  cuDNN SDPA flag error: {e}")

try:
    torch.backends.cuda.enable_flash_sdp(False)
    print("  Flash SDPA disabled (T4 unsupported)")
except Exception as e:
    print(f"  Flash SDPA flag error: {e}")

try:
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(True)
    print("  mem_efficient + math SDPA enabled (fp16 fallbacks)")
except Exception as e:
    print(f"  SDPA enable error: {e}")

# Verify internet for HF download
import socket
try:
    socket.create_connection(("huggingface.co", 443), timeout=5)
    print("  Hugging Face connectivity: OK")
except Exception as e:
    print(f"  ⚠️ Hugging Face connectivity failed: {e}")
    print("     -> Make sure Internet is ON in Kaggle Notebook Settings (right panel)")

print("\n✅ Environment setup and memory optimizations applied!")
print("   If you see 'PYTORCH_CUDA_ALLOC_CONF' warnings, they are harmless - setting took effect for next allocations.")


## 📦 Cell 2 - Install Dependencies

In [ ]:
# Cell 2: Install required packages — Robust Kaggle Fix (with NumPy/SciPy compatibility fix for ImportError _center)
# NOTE: MOSS-TTS v1.5 does NOT require pynini / WeTextProcessing (that is the
# #1 cause of install failures here - they have no prebuilt wheels and are
# only used by other TTS stacks). The real dependency list below is copied
# straight from the official MOSS-TTS pyproject.toml and pinned to avoid
# version drift breaking the remote-code model.
# Make sure "Internet" is ON in Notebook Settings.

import sys, os, subprocess, importlib, torch

print("=== Checking existing environment before install ===")
print(f"Current torch: {torch.__version__} (cuda={torch.version.cuda if hasattr(torch.version,'cuda') else 'unknown'})")
print(f"Python: {sys.version.split()[0]}")

# --- System packages ---
print("\n[1/5] Installing system packages (ffmpeg, libsndfile1)...")
ret = os.system("apt-get update -qq 2>&1 | tail -5")
import shutil
apt_cmd = "DEBIAN_FRONTEND=noninteractive apt-get install -y -qq ffmpeg libsndfile1-dev > /tmp/apt.log 2>&1 && echo 'apt-ok' || echo 'apt-fail'"
print(f"Running: {apt_cmd}")
out = subprocess.getoutput(apt_cmd)
if "apt-ok" not in out:
    out2 = subprocess.getoutput("sudo " + apt_cmd)
    if "apt-ok" in out2:
        print("System packages installed (via sudo)")
    else:
        print("⚠️ apt install may have failed - checking logs:")
        print(open("/tmp/apt.log").read()[-2000:] if os.path.exists("/tmp/apt.log") else out+out2)
        print("Continuing anyway - ffmpeg may already be present")
        if shutil.which("ffmpeg"):
            print("ffmpeg found at", shutil.which("ffmpeg"))
        else:
            print("⚠️ ffmpeg NOT found - audio decoding may fail")
else:
    print("System packages installed")

# --- Helper to pip install with retries ---
def pip_install(pkg, extra_args=""):
    cmd = f"{sys.executable} -m pip install -q --no-cache-dir --upgrade {extra_args} {pkg} 2>&1 | tail -40"
    print(f"\n>>> pip install: {pkg} {extra_args}")
    out = subprocess.getoutput(cmd)
    print(out[-2500:])
    return out

# --- Torch stack ---
from packaging import version as pkg_version
try:
    cur_ver = pkg_version.parse(torch.__version__.split("+")[0])
    need_torch_reinstall = cur_ver < pkg_version.parse("2.4.0") or not torch.cuda.is_available()
except:
    need_torch_reinstall = True

if need_torch_reinstall:
    print("\n[2/5] Installing pinned PyTorch stack (torch / torchaudio / torchcodec, CUDA 12.8)...")
    print("This is a large download (~2GB) - may take 3-5 minutes. Please wait...")
    pip_install("torch==2.9.1+cu128 torchaudio==2.9.1+cu128 torchcodec==0.8.1 --extra-index-url https://download.pytorch.org/whl/cu128", extra_args="--upgrade")
    print("Torch reinstall done - if next cell fails to import torch, do Kernel -> Restart and re-run from Cell 1")
else:
    print(f"\n[2/5] Skipping heavy torch reinstall (current {torch.__version__} is compatible).")
    print("To force reinstall, run: !pip install -q torch==2.9.1+cu128 torchaudio==2.9.1+cu128 torchcodec==0.8.1 --extra-index-url https://download.pytorch.org/whl/cu128")
    try:
        import torchcodec
        print(f"torchcodec present: {torchcodec.__version__}")
    except ImportError:
        print("torchcodec missing - installing...")
        pip_install("torchcodec==0.8.1 --extra-index-url https://download.pytorch.org/whl/cu128")

# --- FIX: NumPy / SciPy / scikit-learn compatibility (ROOT CAUSE of ImportError: _center) ---
# The traceback: transformers -> generation -> sklearn -> scipy -> numpy
#   ImportError: cannot import name '_center' from 'numpy._core.umath'
# Cause: numpy 2.2+ removed private symbol _center, but an older scipy (compiled for numpy 1.x or 2.1)
# is still installed. Pip's default "install -q" without --upgrade keeps stale scipy/sklearn.
# Fix: force-reinstall a KNOWN COMPATIBLE stack.
# We use 2 stacks:
#   Stack A (modern Colab, Python 3.12): numpy==2.1.3, scipy==1.15.3, scikit-learn==1.6.1
#   Stack B (legacy Kaggle fallback):      numpy==1.26.4, scipy==1.13.1, scikit-learn==1.5.2
# We try to detect and install Stack A first, because transformers 5.0 supports numpy 2.x.
# If that still fails, fallback to Stack B.

print("\n[3/5] Fixing numpy/scipy/scikit-learn compatibility (ImportError _center fix)...")

def test_import_chain():
    try:
        # purge cached modules to get fresh import
        for m in list(sys.modules.keys()):
            if m.startswith("numpy") or m.startswith("scipy") or m.startswith("sklearn"):
                # don't delete numpy itself yet, but ensure next import is fresh after reinstall, so skip here
                pass
        import numpy
        import scipy
        import sklearn
        from scipy.sparse import csr_matrix
        from sklearn.metrics import roc_curve
        print(f"  import chain OK: numpy {numpy.__version__}, scipy {scipy.__version__}, sklearn {sklearn.__version__}")
        return True, f"{numpy.__version__}/{scipy.__version__}/{sklearn.__version__}"
    except Exception as e:
        print(f"  import chain FAILED: {e}")
        return False, str(e)

ok, info = test_import_chain()
if not ok:
    print("  -> Need to reinstall compatible stack")

# Force reinstall Stack A (numpy 2.1.3) - preferred for Python 3.12 / Colab / Kaggle 2025+
print("\n  Installing compatible numerical stack A (numpy 2.1.3 + scipy 1.15.3 + scikit-learn 1.6.1)...")
pip_install("'numpy==2.1.3' 'scipy==1.15.3' 'scikit-learn==1.6.1'", extra_args="--force-reinstall --no-cache-dir")

ok, info = test_import_chain()
if not ok:
    print("\n  Stack A still failing, trying Stack B fallback (numpy 1.26.4 + scipy 1.14.1 + scikit-learn 1.5.2)...")
    pip_install("'numpy==1.26.4' 'scipy==1.14.1' 'scikit-learn==1.5.2'", extra_args="--force-reinstall --no-cache-dir")
    ok, info = test_import_chain()

if not ok:
    print("\n  ⚠️ Both stacks failed - trying aggressive purge and reinstall...")
    # Aggressive: uninstall then install
    subprocess.getoutput(f"{sys.executable} -m pip uninstall -y numpy scipy scikit-learn -q 2>&1 | tail -20")
    pip_install("'numpy==1.26.4' 'scipy==1.14.1' 'scikit-learn==1.5.2'", extra_args="--no-cache-dir")
    ok, info = test_import_chain()

if ok:
    print(f"✅ Numerical stack fixed: {info}")
else:
    print(f"❌ Numerical stack still broken: {info}")
    print("   You will need to Kernel -> Restart and try again. If persists, try: !pip install --force-reinstall numpy==1.26.4 scipy==1.14.1 scikit-learn==1.5.2")

# --- Core runtime deps ---
print("\n[4/5] Installing Transformers 5.0.0 and model runtime dependencies...")
# IMPORTANT: We install these with --no-build-isolation? No. Just ensure we don't upgrade numpy away from our fixed version.
# So we first install transformers etc WITHOUT deps that would force numpy upgrade, then re-pin numpy afterwards.
pip_install('"transformers==5.0.0" "accelerate>=1.10.1" soundfile hf-transfer safetensors==0.6.2 orjson==3.11.4 tqdm PyYAML einops librosa tiktoken ninja psutil packaging', extra_args="")

# Re-enforce our fixed numerical stack after transformers install (transformers may have pulled incompatible scipy)
print("\n  Re-enforcing numerical stack after transformers install...")
# Detect which stack succeeded earlier, keep same major version
try:
    import numpy as np
    if np.__version__.startswith("1."):
        pip_install("'numpy==1.26.4' 'scipy==1.14.1' 'scikit-learn==1.5.2'", extra_args="--force-reinstall --no-cache-dir")
    else:
        pip_install("'numpy==2.1.3' 'scipy==1.15.3' 'scikit-learn==1.6.1'", extra_args="--force-reinstall --no-cache-dir")
except:
    pip_install("'numpy==2.1.3' 'scipy==1.15.3' 'scikit-learn==1.6.1'", extra_args="--force-reinstall --no-cache-dir")

print("\n[5/5] Installing Gradio for the web UI...")
pip_install("gradio>=5.23.0")

# --- Verification ---
print("\n=== Verification ===")
for mod in ["torch", "torchaudio", "transformers", "accelerate", "gradio", "soundfile", "librosa", "einops", "numpy", "scipy", "sklearn"]:
    try:
        # Ensure fresh import
        if mod in sys.modules:
            # reimport to get version
            m = importlib.import_module(mod)
            import importlib as _imp
            _imp.reload(m)
        else:
            m = importlib.import_module(mod)
        ver = getattr(m, "__version__", "unknown")
        print(f"✅ {mod}: {ver}")
    except Exception as e:
        print(f"❌ {mod}: {e}")

# Check ffmpeg again
if shutil.which("ffmpeg"):
    print(f"✅ ffmpeg: {shutil.which('ffmpeg')}")
    print(subprocess.getoutput("ffmpeg -version 2>&1 | head -1"))
else:
    print("❌ ffmpeg not found - install may have failed")

# Final test: the exact import that was failing in Cell 3
print("\n=== Final Test: Cell 3 imports ===")
try:
    from transformers import AutoModel, AutoProcessor
    from scipy.sparse import csr_matrix
    from sklearn.metrics import roc_curve
    import numpy
    print(f"✅ SUCCESS: transformers + scipy + sklearn chain works (numpy {numpy.__version__})")
    print("✅ Ready for Cell 3")
except Exception as e:
    import traceback
    print(f"❌ Final test FAILED: {e}")
    traceback.print_exc()
    print("\n💡 Fix: Kernel -> Restart & Clear Outputs, then Run All again")
    print("If still fails, manually run: !pip install --force-reinstall --no-cache-dir 'numpy==1.26.4' 'scipy==1.14.1' 'scikit-learn==1.5.2'")

print("\n✅ Cell 2 completed!")
print("If you reinstalled torch, RECOMMENDED: Kernel -> Restart, then Run All again from top (skips reinstall on second run).")
print("Otherwise, proceed to next cell.")

## 📥 Cell 3 - Load MOSS-TTS Model (Dual GPU Balanced)

In [ ]:
# Cell 3: Load models with cache redirection — Robust Dual-GPU version + NumPy compatibility guard
import os
import gc
import sys
import torch
import traceback

# --- Pre-flight: detect and auto-fix numpy/scipy/sklearn ImportError _center ---
print("=== Pre-flight check for numpy/scipy/sklearn compatibility ===")
try:
    import numpy as np
    import scipy
    import sklearn
    from scipy.sparse import csr_matrix
    from sklearn.metrics import roc_curve
    print(f"✅ Numerical stack OK: numpy {np.__version__}, scipy {scipy.__version__}, sklearn {sklearn.__version__}")
except ImportError as e:
    print(f"⚠️ Detected broken numerical stack: {e}")
    print("This is the ImportError: cannot import name '_center' from 'numpy._core.umath' issue")
    print("Attempting auto-fix...")
    import subprocess
    # Try Stack A then Stack B
    for stack in [
        ["numpy==2.1.3", "scipy==1.15.3", "scikit-learn==1.6.1"],
        ["numpy==1.26.4", "scipy==1.14.1", "scikit-learn==1.5.2"]
    ]:
        print(f"Trying stack: {' '.join(stack)}")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-cache-dir"] + stack)
            # Test again
            # Need to clear modules
            for mod in list(sys.modules.keys()):
                if mod.startswith("numpy") or mod.startswith("scipy") or mod.startswith("sklearn"):
                    del sys.modules[mod]
            import numpy as np
            import scipy
            import sklearn
            from scipy.sparse import csr_matrix
            print(f"✅ Fixed with stack: numpy {np.__version__}, scipy {scipy.__version__}, sklearn {sklearn.__version__}")
            print("⚠️ Please do Kernel -> Restart and run again from Cell 1 for clean state, or proceed - import may work now")
            break
        except Exception as e2:
            print(f"Stack failed: {e2}")
            continue
    else:
        print("❌ Auto-fix failed. Please run manually:")
        print("  !pip install --force-reinstall --no-cache-dir 'numpy==1.26.4' 'scipy==1.14.1' 'scikit-learn==1.5.2'")
        print("  Then Kernel -> Restart & Run All")

# --- Robust HF cache redirection ---
# Use /kaggle/tmp on Kaggle (70GB scratch), else /tmp fallback
if os.path.isdir("/kaggle"):
    base_tmp = "/kaggle/tmp"
else:
    base_tmp = "/tmp"
hf_cache = os.path.join(base_tmp, "hf_cache")
try:
    os.makedirs(hf_cache, exist_ok=True)
    # Also ensure /kaggle/working exists for outputs
    os.makedirs("/kaggle/working", exist_ok=True)
except Exception as e:
    print(f"Could not create {hf_cache}: {e}")
    hf_cache = "/tmp/hf_cache"
    os.makedirs(hf_cache, exist_ok=True)

os.environ["HF_HOME"] = hf_cache
os.environ["HF_HUB_CACHE"] = hf_cache
os.environ["TRANSFORMERS_CACHE"] = hf_cache
os.environ["HF_DATASETS_CACHE"] = hf_cache
# Enable high-speed Hugging Face download (hf-transfer) and mirror endpoint
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
# Use the official Hugging Face endpoint (the hf-mirror.com mirror caused
# intermittent 403/timeout download errors). Remove this if you are in China.
os.environ["HF_ENDPOINT"] = "https://huggingface.co"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
# Also set for huggingface_hub library
os.environ["HUGGINGFACE_HUB_CACHE"] = hf_cache

print(f"HF cache dir: {hf_cache}")
# Show disk space
try:
    import shutil
    total, used, free = shutil.disk_usage(base_tmp)
    print(f"Disk free at {base_tmp}: {free/1e9:.1f} GB")
    total2, used2, free2 = shutil.disk_usage("/")
    print(f"Disk free at /: {free2/1e9:.1f} GB")
except: pass

from transformers import AutoModel, AutoProcessor
# BitsAndBytesConfig is optional - only needed if you do 8bit/4bit quantization (we don't)
try:
    from transformers import BitsAndBytesConfig
except ImportError:
    BitsAndBytesConfig = None

MODEL_ID = "OpenMOSS-Team/MOSS-TTS-v1.5"

# Device plan: Tokenizer on CPU (to save VRAM), Model sharded across both GPUs if available
device_tok = "cpu"
if torch.cuda.is_available():
    device_count = torch.cuda.device_count()
    if device_count >= 2:
        device_model = "cuda:0"  # device_map=auto will shard, but input_ids start on cuda:0
        device_map = "auto"
    elif device_count == 1:
        device_model = "cuda:0"
        device_map = "cuda:0"  # single GPU, no auto sharding needed
        print("Single GPU mode: device_map=cuda:0")
    else:
        device_model = "cpu"
        device_map = "cpu"
else:
    device_model = "cpu"
    device_map = "cpu"

dtype = torch.float16 if torch.cuda.is_available() else torch.float32  # T4 has no bfloat16
print(f"Loading processor and model (dtype={dtype}, device_map={device_map}, device_tok={device_tok})...")
print(f"Model: {MODEL_ID}")

# --- Load processor with retry ---
processor = None
for attempt in range(3):
    try:
        print(f"Attempt {attempt+1}/3: Loading processor...")
        processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
        print("✅ Processor loaded")
        break
    except Exception as e:
        print(f"Attempt {attempt+1} failed: {e}")
        if attempt == 2:
            print(traceback.format_exc())
            raise
        import time; time.sleep(5)

# Keep the processor's audio tokenizer on CPU in float32 precision to save VRAM and prevent bias type mismatch
try:
    if hasattr(processor, "audio_tokenizer") and processor.audio_tokenizer is not None:
        # Some audio_tokenizer implementations are nn.Module, others are custom codec objects
        tok = processor.audio_tokenizer
        if hasattr(tok, "to"):
            try:
                processor.audio_tokenizer = tok.to(device_tok).float()
                print(f"Processor audio tokenizer moved to {device_tok} in float32")
            except Exception as e:
                # Try alternative: .to(device) without .float() then .float() separately
                try:
                    tok = tok.to(device_tok)
                    if hasattr(tok, "float"):
                        tok = tok.float()
                    processor.audio_tokenizer = tok
                    print(f"Processor audio tokenizer moved to {device_tok} (fallback)")
                except Exception as e2:
                    print(f"⚠️ Could not move audio_tokenizer to CPU: {e} / {e2} - keeping on original device (VRAM usage higher but OK)")
        else:
            print("audio_tokenizer has no .to() - keeping as is")
    else:
        print("No audio_tokenizer attribute - skipping CPU move (model may handle internally)")
except Exception as e:
    print(f"⚠️ audio_tokenizer handling error (non-fatal): {e}")

# --- Load main autoregressive model sharded ---
print(f"Loading model weights with dtype={dtype}, device_map={device_map}, attn=sdpa...")
# transformers 5.0 renamed torch_dtype -> dtype, but still supports torch_dtype as alias
# Try dtype first, fallback to torch_dtype
model = None
load_kwargs = dict(
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)
# Handle dtype param name compatibility
try:
    import inspect
    sig = inspect.signature(AutoModel.from_pretrained)
    if "dtype" in sig.parameters:
        load_kwargs["dtype"] = dtype
        print("Using `dtype` param (transformers 5.x)")
    else:
        load_kwargs["torch_dtype"] = dtype
        print("Using `torch_dtype` param (transformers 4.x compat)")
except:
    load_kwargs["torch_dtype"] = dtype

if device_map != "cpu":
    load_kwargs["device_map"] = device_map
else:
    load_kwargs["device_map"] = "cpu"

# Retry loop for model download (HF hub sometimes 429/403)
for attempt in range(3):
    try:
        print(f"Attempt {attempt+1}/3: from_pretrained...")
        model = AutoModel.from_pretrained(MODEL_ID, **load_kwargs).eval()
        print("✅ Model loaded and sharded")
        break
    except Exception as e:
        print(f"Attempt {attempt+1} failed: {e}")
        traceback.print_exc()
        if "No space left" in str(e) or "disk" in str(e).lower():
            print("❌ Disk full! Check that HF_HOME is /kaggle/tmp/hf_cache and free space >20GB")
            print("   Run: !du -sh /kaggle/tmp/*  and  !df -h")
        if attempt == 2:
            raise
        import time; time.sleep(8)

# --- Apply Multi-GPU device safety monkeypatches to handle accelerate sharding ---
import types
try:
    def fixed_get_input_embeddings(self, input_ids: torch.LongTensor) -> torch.Tensor:
        # Handle sharded model where embed layers live on different GPUs
        base_embed = self.language_model.get_input_embeddings()
        # base_embed is on some GPU, move first token embeddings there
        target_device = next(base_embed.parameters()).device
        inputs_embeds = base_embed(input_ids[..., 0].to(target_device)).to(dtype=dtype)
        for i, embed_layer in enumerate(self.emb_ext):
            vq_device = next(embed_layer.parameters()).device
            idx = input_ids[..., i + 1].to(vq_device)
            val = embed_layer(idx).to(device=inputs_embeds.device, dtype=inputs_embeds.dtype)
            inputs_embeds = inputs_embeds + val
        return inputs_embeds

    model.get_input_embeddings = types.MethodType(fixed_get_input_embeddings, model)
    print("✅ Patched get_input_embeddings for multi-GPU sharding")
except Exception as e:
    print(f"⚠️ get_input_embeddings patch failed (non-fatal, single-GPU may still work): {e}")
    traceback.print_exc()

try:
    original_forward = model.forward
    def fixed_forward(self, *args, **kwargs):
        input_ids = kwargs.get("input_ids", None)
        if input_ids is None and len(args) > 0:
            input_ids = args[0]
        outputs = original_forward(*args, **kwargs)
        if outputs.logits is not None:
            try:
                target_device = input_ids.device if input_ids is not None else next(self.parameters()).device
                outputs.logits = [logit.to(target_device) for logit in outputs.logits]
            except Exception as e:
                # If sharding moves logits, keep as is
                pass
        return outputs
    model.forward = types.MethodType(fixed_forward, model)
    print("✅ Patched forward for logit device safety")
except Exception as e:
    print(f"⚠️ forward patch failed (non-fatal): {e}")

# --- Apply numerical stability monkeypatch for sample_token to prevent NaN/inf CUDA assertions (T4 fp16) ---
import torch.nn.functional as F
target_modules = []
for name, module in list(sys.modules.items()):
    if "inference_utils" in name or "modeling_moss_tts" in name:
        target_modules.append((name, module))
# Also try direct import if not yet loaded
if not target_modules:
    for cand in ["modeling_moss_tts", "moss_tts", "inference_utils"]:
        try:
            m = importlib.import_module(cand)
            target_modules.append((cand, m))
        except: pass
        # try to find file in HF cache
    # Search sys.modules again after model load, sometimes nested
    for name, module in list(sys.modules.items()):
        if "moss" in name.lower() or "inference" in name.lower():
            if (name, module) not in target_modules:
                target_modules.append((name, module))

if target_modules:
    # Get helpers from the actual inference_utils module
    utils_module = None
    for name, module in target_modules:
        if "inference_utils" in name:
            utils_module = module
            break
    if utils_module is None:
        utils_module = target_modules[0][1]
        
    # Only patch if utils has expected helpers
    has_helpers = all(hasattr(utils_module, x) for x in ["apply_repetition_penalty_delay_pattern", "apply_top_k", "apply_top_p_optimized"])
    if not has_helpers:
        print(f"⚠️ utils_module {utils_module} missing expected helpers - sample_token patch will use fallback implementations")
        # Define fallbacks
        def fallback_apply_top_k(logits, k):
            if k is None or k <=0:
                return logits
            # Simple top_k masking
            topk_vals, _ = torch.topk(logits, k, dim=-1)
            kth = topk_vals[..., -1, None]
            mask = logits < kth
            logits = logits.masked_fill(mask, float('-inf'))
            return logits
        def fallback_apply_top_p(logits, p):
            if p is None or p>=1.0:
                return logits
            sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
            probs = F.softmax(sorted_logits.float(), dim=-1)
            cum = torch.cumsum(probs, dim=-1)
            mask = cum - probs > p
            sorted_logits[mask] = float('-inf')
            # scatter back
            # logits is 2D [batch*seq, vocab], we need to unsort
            # Simpler: just return sorted_logits sorted back is complex, so use original logits masking via sorted order
            # For fallback, just return as is if not trivial
            return logits
        def fallback_repetition(logits, prev, penalty):
            return logits
        # Monkey fix utils_module to have them
        if not hasattr(utils_module, "apply_top_k"):
            utils_module.apply_top_k = fallback_apply_top_k
        if not hasattr(utils_module, "apply_top_p_optimized"):
            utils_module.apply_top_p_optimized = fallback_apply_top_p
        if not hasattr(utils_module, "apply_repetition_penalty_delay_pattern"):
            utils_module.apply_repetition_penalty_delay_pattern = fallback_repetition
        
    def fixed_sample_token(logits, prev_tokens=None, repetition_penalty=1.0, top_p=None, top_k=None, do_sample=True):
        # Fix NaN/inf for fp16 on T4
        if torch.isnan(logits).any() or torch.isinf(logits).any():
            logits = torch.nan_to_num(logits, nan=0.0, posinf=1e4, neginf=-1e4)
            
        vocab_size = logits.size(-1)
        if prev_tokens is not None and repetition_penalty != 1.0:
            try:
                logits = utils_module.apply_repetition_penalty_delay_pattern(
                    logits, prev_tokens, repetition_penalty
                )
            except Exception as e:
                # Fallback: skip penalty if fails
                pass
            
        if not do_sample:
            return torch.argmax(logits, dim=-1)
            
        original_shape = logits.shape
        reshaped_logits = logits.view(-1, vocab_size)
        
        # Check for rows that are entirely -inf (softmax will output NaNs on them)
        try:
            all_neginf = (reshaped_logits == float('-inf')).all(dim=-1)
            if all_neginf.any():
                reshaped_logits[all_neginf, 0] = 0.0
        except: pass
            
        if top_k is not None and top_k > 0:
            try:
                reshaped_logits = utils_module.apply_top_k(reshaped_logits, top_k)
            except: pass
            
        if top_p is not None and top_p < 1.0:
            try:
                reshaped_logits = utils_module.apply_top_p_optimized(reshaped_logits, top_p)
            except: pass
            
        # Cast to float32 before softmax and multinomial for numerical stability in float16/bfloat16
        try:
            probs = F.softmax(reshaped_logits.float(), dim=-1)
        except Exception as e:
            # Fallback: nan_to_num then softmax
            reshaped_logits = torch.nan_to_num(reshaped_logits.float(), nan=0.0, posinf=1e4, neginf=-1e4)
            probs = F.softmax(reshaped_logits, dim=-1)
        
        if torch.isnan(probs).any():
            probs = torch.nan_to_num(probs, nan=0.0)
            sums = probs.sum(dim=-1, keepdim=True)
            sums[sums == 0.0] = 1.0
            probs = probs / sums
            
        sums = probs.sum(dim=-1)
        zero_sums = sums == 0.0
        if zero_sums.any():
            probs[zero_sums, 0] = 1.0
            
        try:
            next_tokens = torch.multinomial(probs, num_samples=1)
        except Exception as e:
            # Fallback to argmax if multinomial fails (e.g. probs still invalid)
            print(f"multinomial failed {e}, falling back to argmax")
            next_tokens = torch.argmax(probs, dim=-1, keepdim=True)
        return next_tokens.view(original_shape[:-1])
        
    patched = 0
    for name, module in target_modules:
        if hasattr(module, "sample_token"):
            try:
                module.sample_token = fixed_sample_token
                print(f"✅ Patched sample_token in module: {name}")
                patched += 1
            except Exception as e:
                print(f"Failed to patch {name}: {e}")
    if patched == 0:
        print(f"⚠️ No sample_token found to patch in {len(target_modules)} modules: {[n for n,_ in target_modules]}")
        print("   Model will use original sampling - may get NaN errors on T4 fp16 if unlucky")
else:
    print("⚠️ Could not find target modules to patch (inference_utils/modeling_moss_tts not yet imported).")
    print("   This is OK if model hasn't been imported fully - original sampling will be used")
    print("   If you get 'probability tensor contains nan/inf' errors, restart and ensure Cell 3 completed without errors")

# Final memory report
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        try:
            alloc = torch.cuda.memory_allocated(i)/1e9
            reserved = torch.cuda.memory_reserved(i)/1e9
            print(f"GPU {i} memory: {alloc:.2f} GB allocated, {reserved:.2f} GB reserved")
        except: pass
    torch.cuda.empty_cache()
    gc.collect()

print("\n✅ Models loaded successfully, sharded, and device-safety patched!")
print("   Ready for Cell 4 (Gradio UI). If you see OOM later, try lowering max_new_tokens or using single GPU.")

## 🎛️ Cell 4 - Launch Gradio Web UI

In [ ]:
# Cell 4: Launch Gradio App with Soft Theme and Fast Preset — Robust Kaggle Version
import time
import gc
import os
import sys
import tempfile
import traceback
from datetime import datetime

# Ensure torch is available even if Cell 3 was re-run standalone or after kernel restart
import torch
import torchaudio
import gradio as gr
from transformers import GenerationConfig

# Optional: soundfile fallback for torchaudio.save failures (common on Kaggle with codec mismatches)
try:
    import soundfile as sf
    HAS_SOUNDFILE = True
except ImportError:
    HAS_SOUNDFILE = False

# Subclass GenerationConfig to match custom MOSS-TTS Delay configuration parameters
class DelayGenerationConfig(GenerationConfig):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.layers = kwargs.get("layers", [{} for _ in range(32)])
        self.do_samples = kwargs.get("do_samples", None)
        self.n_vq_for_inference = 32

# Supported languages list
LANGUAGES = [
    "Auto (omit)", "English", "Chinese", "Japanese", "Korean", "French", "German",
    "Spanish", "Italian", "Portuguese", "Russian", "Arabic", "Cantonese",
    "Vietnamese", "Thai", "Turkish", "Hindi", "Indonesian", "Malay",
    "Dutch", "Swedish", "Polish", "Danish", "Finnish", "Norwegian",
    "Czech", "Greek", "Hungarian", "Romanian", "Slovak", "Ukrainian",
    "Hebrew"
]

def to_device(obj, device):
    """Recursively map PyTorch tensors inside nested structures to target device"""
    if obj is None:
        return None
    if isinstance(obj, torch.Tensor):
        return obj.to(device)
    elif isinstance(obj, list):
        return [to_device(x, device) for x in obj]
    elif isinstance(obj, dict):
        return {k: to_device(v, device) for k, v in obj.items()}
    elif isinstance(obj, tuple):
        return tuple(to_device(x, device) for x in obj)
    return obj

ZH_TOKENS_PER_CHAR = 3.098411951313033
EN_TOKENS_PER_CHAR = 0.8673376262755219

def detect_text_language(text: str) -> str:
    import re
    zh_chars = len(re.findall(r"[\u4e00-\u9fff]", text))
    en_chars = len(re.findall(r"[A-Za-z]", text))
    if zh_chars == 0 and en_chars == 0:
        return "en"
    return "zh" if zh_chars >= en_chars else "en"

def supports_duration_control(mode_with_reference: str) -> bool:
    return mode_with_reference not in ["Continuation", "Continuation + Clone"]

def estimate_duration_tokens(text: str) -> tuple[str, int, int, int]:
    normalized = text or ""
    effective_len = max(len(normalized), 1)
    language = detect_text_language(normalized)
    factor = ZH_TOKENS_PER_CHAR if language == "zh" else EN_TOKENS_PER_CHAR
    default_tokens = max(1, int(effective_len * factor))
    min_tokens = max(1, int(default_tokens * 0.5))
    max_tokens = max(min_tokens, int(default_tokens * 1.5))
    return language, default_tokens, min_tokens, max_tokens

def update_duration_controls(
    enabled: bool,
    text: str,
    current_tokens: float | int | None,
    mode_with_reference: str,
):
    if not supports_duration_control(mode_with_reference):
        return (
            gr.update(visible=False),
            "Duration control is disabled for Continuation modes.",
            gr.update(value=False, interactive=False),
        )

    checkbox_update = gr.update(interactive=True)
    if not enabled:
        return gr.update(visible=False), "Duration control is disabled.", checkbox_update

    language, default_tokens, min_tokens, max_tokens = estimate_duration_tokens(text)
    if current_tokens is None or int(current_tokens) == 1:
        slider_value = default_tokens
    else:
        slider_value = int(current_tokens)
        slider_value = max(min_tokens, min(max_tokens, slider_value))

    language_label = "Chinese" if language == "zh" else "English"
    hint = (
        f"Duration control enabled | detected language: {language_label} | "
        f"default={default_tokens}, range=[{min_tokens}, {max_tokens}]"
    )
    return (
        gr.update(
            visible=True,
            minimum=min_tokens,
            maximum=max_tokens,
            value=slider_value,
            step=1,
        ),
        hint,
        checkbox_update,
    )

def render_mode_hint(reference_audio: str | None, mode_with_reference: str):
    # Gradio returns "" when no file, not None
    if not reference_audio:
        return "Current mode: **Direct Generation** (no reference audio uploaded)"
    if mode_with_reference == "Clone":
        return "Current mode: **Clone** (speaker timbre will be cloned from the reference audio)"
    return f"Current mode: **{mode_with_reference}**  \n> Continuation mode is active. Make sure the reference audio transcript is prepended to the input text."

def generate_speech(
    text,
    language,
    reference_audio,
    mode_with_reference,
    speed=1.0,
    text_temp=1.2,
    text_top_p=1.0,
    text_top_k=50,
    audio_temp=1.7,
    audio_top_p=0.9,
    audio_top_k=25,
    audio_repetition_penalty=1.0,
    duration_control_enabled=False,
    expected_tokens_val=1,
    progress=gr.Progress()
):
    """
    Optimized speech generation backend running model.generate with auto-sharded FP16
    and audio tokenizer decoding on CPU. Supports Clone, Continuation, and Continuation + Clone modes.
    """
    n_vq = 32
    max_new_tokens_val = 32768

    if not text or not text.strip():
        return None, "Error: Text input cannot be empty."
    
    # Ensure model/processor exist (helpful error if Cell 3 not run)
    if 'processor' not in globals() or 'model' not in globals():
        return None, "❌ Error: Model not loaded. Please run Cell 3 (Load MOSS-TTS Model) first and wait for 'Models loaded successfully'."
    
    status_log = []
    start_time = time.time()
    
    try:
        progress(0, desc="Preprocessing input...")
        status_log.append("🔄 Step 1: Preprocessing input...")
        
        # Normalize reference_audio: Gradio returns "" or None when empty
        if reference_audio == "" or reference_audio is None:
            reference_audio = None
        else:
            # Validate file exists
            if not os.path.exists(reference_audio):
                return None, f"❌ Error: Reference audio not found at {reference_audio}"
        
        # Determine if expected tokens is active
        duration_enabled = bool(duration_control_enabled and supports_duration_control(mode_with_reference))
        expected_tokens = int(expected_tokens_val) if duration_enabled else None
        
        lang_code = None if language == "Auto (omit)" else language
        
        status_log.append(f"  Language Tag: {language}")
        if reference_audio:
            status_log.append(f"  Voice reference: {reference_audio}")
            status_log.append(f"  Reference Mode: {mode_with_reference}")
        else:
            status_log.append("  Mode: Direct Generation (No reference audio)")
            
        if expected_tokens is not None:
            status_log.append(f"  Expected tokens (duration control): {expected_tokens}")
        status_log.append(f"  Max new tokens (budget): {max_new_tokens_val}")
        
        user_kwargs = {"text": text, "language": lang_code}
        if expected_tokens is not None:
            user_kwargs["tokens"] = int(expected_tokens)
        
        # Format the conversations list and determine generation mode
        if not reference_audio:
            message = processor.build_user_message(**user_kwargs)
            conversations = [[message]]
            mode = "generation"
        else:
            if mode_with_reference == "Clone":
                clone_kwargs = dict(user_kwargs)
                clone_kwargs["reference"] = [reference_audio]
                message = processor.build_user_message(**clone_kwargs)
                conversations = [[message]]
                mode = "generation"
            elif mode_with_reference == "Continuation":
                message = processor.build_user_message(**user_kwargs)
                assistant_message = processor.build_assistant_message(audio_codes_list=[reference_audio])
                conversations = [[message, assistant_message]]
                mode = "continuation"
            else: # Continuation + Clone
                continue_clone_kwargs = dict(user_kwargs)
                continue_clone_kwargs["reference"] = [reference_audio]
                message = processor.build_user_message(**continue_clone_kwargs)
                assistant_message = processor.build_assistant_message(audio_codes_list=[reference_audio])
                conversations = [[message, assistant_message]]
                mode = "continuation"
        
        # Tokenize and format inputs
        try:
            batch = processor(conversations, mode=mode, n_vq=32)
        except Exception as e:
            return None, f"❌ Error in processor tokenization: {e}\n{traceback.format_exc()}"
        
        # batch can be dict-like or BatchEncoding; handle both
        try:
            if isinstance(batch, dict):
                input_ids = batch["input_ids"]
                attention_mask = batch["attention_mask"]
            else:
                input_ids = batch.input_ids
                attention_mask = batch.attention_mask
        except Exception as e:
            return None, f"❌ Error extracting input_ids from batch: {e}\nBatch keys: {batch.keys() if isinstance(batch, dict) else dir(batch)}"
        
        # Move to device_model (handle already CUDA tensors)
        try:
            # device_model is defined in Cell 3 globals
            _device_model = globals().get("device_model", "cuda:0" if torch.cuda.is_available() else "cpu")
            input_ids = input_ids.to(_device_model)
            attention_mask = attention_mask.to(_device_model)
            _device_tok = globals().get("device_tok", "cpu")
        except NameError:
            _device_model = "cuda:0" if torch.cuda.is_available() else "cpu"
            _device_tok = "cpu"
            input_ids = input_ids.to(_device_model)
            attention_mask = attention_mask.to(_device_model)
        except Exception as e:
            return None, f"❌ Error moving inputs to device: {e}\n{traceback.format_exc()}"
        
        # Fix temperature boundaries to prevent division by zero in sampling
        if audio_temp == 0.0:
            audio_temp = 0.001
        if text_temp == 0.0:
            text_temp = 0.001
        
        # Clear VRAM cache before running generation
        if torch.cuda.is_available():
            try:
                torch.cuda.empty_cache()
                gc.collect()
            except: pass
            
        progress(0.3, desc="Generating speech tokens (Autoregression)...")
        status_log.append(f"🧠 Step 2: Generating speech tokens on {_device_model}...")
        status_log.append(f"   text_temp={text_temp}, text_top_p={text_top_p}, text_top_k={text_top_k}, audio_temp={audio_temp}, audio_top_p={audio_top_p}, audio_top_k={audio_top_k}")
        
        with torch.no_grad():
            try:
                outputs = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=int(max_new_tokens_val),
                    text_temperature=float(text_temp),
                    text_top_p=float(text_top_p),
                    text_top_k=int(text_top_k),
                    audio_temperature=float(audio_temp),
                    audio_top_p=float(audio_top_p),
                    audio_top_k=int(audio_top_k),
                    audio_repetition_penalty=float(audio_repetition_penalty)
                )
            except RuntimeError as e:
                err_str = str(e)
                if "out of memory" in err_str.lower() or "OOM" in err_str:
                    # Try to free and give actionable advice
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                        gc.collect()
                    return None, f"❌ CUDA OOM: {e}\n\n💡 Fixes:\n- Try shorter text (split into sentences)\n- Reduce max_new_tokens (currently {max_new_tokens_val}) - edit Cell 4 code\n- Ensure you selected T4 x2 (dual GPU) in Kaggle Settings\n- Restart kernel and run again"
                else:
                    raise
            
        gen_time = time.time() - start_time
        status_log.append(f"  Generated tokens in {gen_time:.2f} seconds.")
        
        progress(0.8, desc="Decoding audio tokens...")
        status_log.append(f"🔊 Step 3: Moving tokens to {_device_tok} and decoding audio...")
        
        # Move output tokens list safely to device_tok
        try:
            outputs_tok = to_device(outputs, _device_tok)
        except Exception as e:
            # If to_device fails, try direct
            print(f"to_device failed {e}, trying direct")
            outputs_tok = outputs
            
        try:
            decoded_messages = processor.decode(outputs_tok)
        except Exception as e:
            return None, f"❌ Error decoding outputs: {e}\n{traceback.format_exc()}\nOutputs type: {type(outputs_tok)}, len: {len(outputs_tok) if hasattr(outputs_tok,'__len__') else 'N/A'}"
            
        if not decoded_messages:
            return None, "Error: Failed to decode output tokens from the model."
            
        # Extract the stereo waveform with empty check safety
        try:
            first_msg = decoded_messages[0]
            codes = getattr(first_msg, "audio_codes_list", None)
            if codes is None:
                # try dict access
                codes = first_msg.get("audio_codes_list") if isinstance(first_msg, dict) else None
            if not codes:
                # Try to show generated text for debugging
                try:
                    gen_text_ids = outputs_tok[0][1][:, 0] if (outputs_tok and len(outputs_tok) > 0 and hasattr(outputs_tok[0], '__getitem__')) else None
                    gen_text_decoded = processor.tokenizer.decode(gen_text_ids[0]) if gen_text_ids is not None else "Unknown"
                except Exception as e:
                    gen_text_decoded = f"Could not decode text: {e}"
                err_msg = (
                    "❌ Error: Model generated text but failed to synthesize any audio codes.\n"
                    f"Generated Text Output: {gen_text_decoded}\n"
                    "Please verify that prompt language matches, check hyperparameters (e.g. reduce temperatures), "
                    "or try a different input text.\n"
                    "Tip: For Continuation modes, ensure reference transcript is prepended to input text."
                )
                return None, err_msg
            audio = codes[0]
        except Exception as e:
            return None, f"❌ Error extracting audio codes: {e}\n{traceback.format_exc()}"
        
        # Ensure dimensions match: [channels, samples]
        try:
            if audio.ndim == 1:
                audio = audio.unsqueeze(0)
            # Ensure stereo: if mono, keep mono (model outputs stereo 48kHz normally)
            if audio.ndim != 2:
                print(f"Warning: audio ndim={audio.ndim}, shape={audio.shape}, fixing")
                audio = audio.reshape(1, -1) if audio.numel()>0 else audio
        except Exception as e:
            print(f"Audio shape fix warning: {e}")
            
        # Clear tensor allocation references
        try:
            del outputs, input_ids, attention_mask, batch, decoded_messages
        except: pass
        if torch.cuda.is_available():
            try:
                torch.cuda.empty_cache()
                gc.collect()
            except: pass
            
        # Adjust playback speed if requested via resampling
        if speed != 1.0:
            progress(0.9, desc="Adjusting speech speed...")
            status_log.append(f"🏃 Speed adjustment: Resampling audio to {speed}x...")
            try:
                sample_rate = processor.model_config.sampling_rate
                new_sample_rate = int(sample_rate * speed)
                resampler = torchaudio.transforms.Resample(
                    orig_freq=sample_rate,
                    new_freq=new_sample_rate
                )
                audio_resampled = resampler(audio).squeeze(0)
                resampler_back = torchaudio.transforms.Resample(
                    orig_freq=new_sample_rate,
                    new_freq=sample_rate
                )
                # Handle mono vs stereo for resampler
                if audio.dim() == 2:
                    # resample per channel? torchaudio expects [channels, time]
                    audio_resampled = resampler(audio)
                    audio = resampler_back(audio_resampled)
                else:
                    audio = resampler_back(audio_resampled.unsqueeze(0))
            except Exception as e:
                status_log.append(f"⚠️ Speed adjustment failed: {e}, using original audio")
            
        progress(0.95, desc="Saving audio file...")
        # Save generated audio to a temporary file - use /kaggle/working if exists else tempfile
        try:
            if os.path.isdir("/kaggle/working"):
                tmpdir = "/kaggle/working"
            else:
                tmpdir = tempfile.gettempdir()
            temp_file = tempfile.NamedTemporaryFile(suffix=".wav", delete=False, dir=tmpdir)
            output_path = temp_file.name
            temp_file.close()
        except:
            temp_file = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
            output_path = temp_file.name
            temp_file.close()
        
        try:
            sampling_rate = processor.model_config.sampling_rate
        except:
            sampling_rate = 48000
            print(f"Could not get sampling_rate from config, using {sampling_rate}")
        
        # Save with torchaudio, fallback to soundfile
        try:
            torchaudio.save(
                output_path, 
                audio.detach().cpu().to(torch.float32), 
                sampling_rate
            )
        except Exception as e:
            print(f"torchaudio.save failed {e}, trying soundfile fallback")
            if HAS_SOUNDFILE:
                try:
                    # soundfile expects [samples, channels]
                    audio_np = audio.detach().cpu().to(torch.float32).numpy()
                    # transpose if [channels, samples]
                    if audio_np.ndim == 2:
                        audio_np = audio_np.T
                    sf.write(output_path, audio_np, sampling_rate)
                    print("Saved via soundfile")
                except Exception as e2:
                    return None, f"❌ Failed to save audio via both torchaudio and soundfile: {e} / {e2}\n{traceback.format_exc()}"
            else:
                return None, f"❌ torchaudio.save failed and soundfile not available: {e}\n{traceback.format_exc()}"
        
        try:
            duration = audio.shape[-1] / sampling_rate
        except:
            duration = 0
        rtf = gen_time / duration if duration > 0 else 0
        
        status_log.append(f"✅ Success: Generated {duration:.1f}s of audio in {gen_time:.1f}s (RTF: {rtf:.2f}x) at {sampling_rate} Hz!")
        if duration > 0 and rtf > 1:
            status_log.append(f"   Note: RTF>1 means slower than realtime - expected on T4 for long texts. Try shorter sentences for faster.")
        progress(1.0, desc="Done!")
        return output_path, "\n".join(status_log)
        
    except Exception as e:
        err_msg = f"❌ Error during generation: {str(e)}\n{traceback.format_exc()}"
        # Add hint for common errors
        if "probability tensor contains" in str(e) and "nan" in str(e).lower():
            err_msg += "\n💡 Hint: This is a known fp16 NaN issue on T4. Try: reduce audio_temp (e.g. 1.2), increase top_p, or re-run. The patched sampling should prevent this - ensure Cell 3 completed with 'Patched sample_token'."
        if "CUDA" in str(e) and "assert" in str(e):
            err_msg += "\n💡 Hint: CUDA assertion - try lowering temperatures or restarting kernel."
        return None, err_msg

# Branded CSS
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.btn-row { display: flex; justify-content: center; gap: 10px; flex-wrap: wrap; }
.social-btn { display: inline-flex; align-items: center; justify-content: center; min-width: 150px; padding: 10px 18px; border-radius: 10px; font-weight: 700; font-size: 13px; text-decoration: none; color: white; white-space: nowrap; transition: transform 0.2s, box-shadow 0.2s; }
.social-btn:hover { transform: translateY(-2px); box-shadow: 0 6px 16px rgba(0,0,0,0.3); }
.yt-btn  { background: #FF0000; box-shadow: 0 4px 12px rgba(255,0,0,0.3); }
.x-btn   { background: #000000; box-shadow: 0 4px 12px rgba(0,0,0,0.25); }
.sup-btn { background: linear-gradient(135deg,#f6d365,#fda085); box-shadow: 0 4px 12px rgba(253,160,133,0.35); }
button.primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#stop-btn { background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#clear-btn { background: linear-gradient(135deg, #6b7280 0%, #374151 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
.footer { text-align: center; padding: 20px; margin-top: 30px; border-top: 2px solid #e5e7eb; color: #6b7280; }
"""

with gr.Blocks(title="MOSS-TTS v1.5 - AIQUEST Academy", theme=gr.themes.Soft(), css=custom_css) as demo:
    # Branded Header Component
    gr.HTML("""
    <div class="brand-header">
      <div class="brand-title">🎤 MOSS-TTS v1.5 - Foundation Model</div>
      <div class="brand-subtitle">Created by <strong>AIQUEST Academy</strong> &nbsp;|&nbsp; Kaggle T4 x2 GPU Edition · Sharded FP16 Precision</div>
      <div class="btn-row">
        <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn yt-btn">▶ Subscribe on YouTube</a>
        <a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a>
        <a href="https://aiquest.site" target="_blank" class="social-btn sup-btn">❤️ Support My Work</a>
      </div>
    </div>
    """)

    gr.HTML("<p style='text-align: center; margin-top: 10px;'>High-Fidelity 48 kHz stereo Speech Synthesis and zero-shot Voice Cloning (Sharded FP16 Precision)</p>")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📥 Input Configuration")
            text_input = gr.Textbox(
                label="Synthesize Text",
                placeholder="Type the text you want to speak. Support inline pauses using e.g., [pause 1.5s] syntax. In continuation modes, prepend reference audio transcript.",
                lines=4,
                value="This is MOSS text-to-speech, running on Kaggle dual T4 GPU free tier. Please subscribe to AIQUEST for more free stuff!"
            )
            lang_input = gr.Dropdown(
                choices=LANGUAGES,
                label="Language Tag (Optional)",
                value="Auto (omit)"
            )
            ref_audio_input = gr.Audio(
                label="Reference Audio (Optional, for zero-shot Voice Cloning or Continuation)",
                type="filepath"
            )
            mode_with_reference = gr.Radio(
                choices=["Clone", "Continuation", "Continuation + Clone"],
                value="Clone",
                label="Mode with Reference Audio",
                info="If no reference audio is uploaded, Direct Generation will be used automatically. Note: Continuation modes require reference transcript at start of text."
            )
            mode_hint = gr.Markdown("Current mode: **Direct Generation** (no reference audio uploaded)")
            
            with gr.Accordion("⚙️ Advanced Settings", open=False):
                with gr.Row():
                    speed_slider = gr.Slider(
                        minimum=0.5, maximum=2.0, value=1.0, step=0.1,
                        label="Speed Factor"
                    )
                
                duration_control_enabled = gr.Checkbox(
                    value=False,
                    label="Enable Duration Control (Expected Audio Tokens)"
                )
                expected_tokens = gr.Slider(
                    minimum=1,
                    maximum=1,
                    step=1,
                    value=1,
                    label="expected_tokens",
                    visible=False
                )
                duration_hint = gr.Markdown("Duration control is disabled.")

                with gr.Row():
                    text_temp_slider = gr.Slider(minimum=0.1, maximum=2.0, value=1.5, step=0.1, label="text_temp")
                    text_top_p_slider = gr.Slider(minimum=0.1, maximum=1.0, value=1.0, step=0.05, label="text_top_p")
                    text_top_k_slider = gr.Slider(minimum=1, maximum=100, value=50, step=1, label="text_top_k")
                with gr.Row():
                    audio_temp_slider = gr.Slider(minimum=0.1, maximum=3.0, value=1.7, step=0.05, label="audio_temp")
                    audio_top_p_slider = gr.Slider(minimum=0.1, maximum=1.0, value=0.9, step=0.01, label="audio_top_p")
                with gr.Row():
                    audio_top_k_slider = gr.Slider(minimum=1, maximum=200, value=25, step=1, label="audio_top_k")
                    audio_rep_pen_slider = gr.Slider(minimum=0.8, maximum=2.0, value=1.0, step=0.05, label="audio_repetition_penalty")

            # Gradio Buttons (Always 3: Generate, Stop, Clear)
            with gr.Row():
                gen_btn = gr.Button("🔊 Generate Audio", variant="primary", size="lg", elem_id="gen-btn")
                stop_btn = gr.Button("🛑 Stop", variant="secondary", size="lg", elem_id="stop-btn")
                clear_btn = gr.Button("🗑️ Clear", variant="secondary", size="lg", elem_id="clear-btn")

        with gr.Column(scale=1):
            gr.Markdown("### 📤 Output Results")
            audio_output = gr.Audio(
                label="Generated Speech",
                interactive=False
            )
            log_output = gr.Textbox(
                label="Process Log",
                lines=14,
                interactive=False
            )

    # Branded Footer
    gr.HTML(
        "<div class='footer'>"
        "<p style='margin:0; text-align:center'>© 2026 AIQUEST Academy. Powered by OpenMOSS-Team MOSS-TTS v1.5.</p>"
        "<p style='margin:4px 0 0 0; font-size:12px; text-align:center'>Kaggle T4 x2: If audio fails, try lowering audio_temp to 1.2, or check that Input Text language matches Language Tag.</p>"
        "</div>"
    )

    # Wire up reference audio and mode hint renders
    ref_audio_input.change(
        fn=render_mode_hint,
        inputs=[ref_audio_input, mode_with_reference],
        outputs=[mode_hint]
    )
    mode_with_reference.change(
        fn=render_mode_hint,
        inputs=[ref_audio_input, mode_with_reference],
        outputs=[mode_hint]
    )

    # Wire up duration control dynamic calculations
    duration_control_enabled.change(
        fn=update_duration_controls,
        inputs=[duration_control_enabled, text_input, expected_tokens, mode_with_reference],
        outputs=[expected_tokens, duration_hint, duration_control_enabled],
    )
    text_input.change(
        fn=update_duration_controls,
        inputs=[duration_control_enabled, text_input, expected_tokens, mode_with_reference],
        outputs=[expected_tokens, duration_hint, duration_control_enabled],
    )
    mode_with_reference.change(
        fn=update_duration_controls,
        inputs=[duration_control_enabled, text_input, expected_tokens, mode_with_reference],
        outputs=[expected_tokens, duration_hint, duration_control_enabled],
    )

    # Wire up button event listeners
    gen_event = gen_btn.click(
        fn=generate_speech,
        inputs=[
            text_input, lang_input, ref_audio_input, mode_with_reference, speed_slider,
            text_temp_slider, text_top_p_slider, text_top_k_slider,
            audio_temp_slider, audio_top_p_slider, audio_top_k_slider, audio_rep_pen_slider,
            duration_control_enabled, expected_tokens
        ],
        outputs=[audio_output, log_output]
    )
    
    stop_btn.click(
        fn=None,
        cancels=[gen_event]
    )
    
    # Clear now matches actual slider defaults (text_temp 1.5, etc)
    clear_btn.click(
        fn=lambda: (None, "", "This is MOSS text-to-speech, running on Kaggle dual T4 GPU free tier. Please subscribe to AIQUEST for more free stuff!", "Auto (omit)", None, "Clone", 1.0, 1.5, 1.0, 50, 1.7, 0.9, 25, 1.0, False, 1),
        inputs=[],
        outputs=[
            audio_output, log_output, text_input, lang_input, ref_audio_input, mode_with_reference, speed_slider,
            text_temp_slider, text_top_p_slider, text_top_k_slider,
            audio_temp_slider, audio_top_p_slider, audio_top_k_slider, audio_rep_pen_slider,
            duration_control_enabled, expected_tokens
        ]
    )

# Launch Gradio interface - Kaggle-compatible
# On Kaggle, share=True works via Gradio's tunnel (requires Internet ON).
# Fallback to inline if share fails.
print("Launching Gradio... (if this hangs, check that Internet is ON in Kaggle Settings)")
try:
    demo.queue(max_size=3)
    # Preferred: share + 0.0.0.0 for Kaggle proxy
    demo.launch(share=True, server_name="0.0.0.0", server_port=7860, show_error=True, allowed_paths=["/kaggle/tmp", "/tmp", "/kaggle/working"])
except Exception as e:
    print(f"share=True launch failed: {e}")
    print("Retrying with share=False (use Kaggle's preview on 7860)...")
    try:
        demo.launch(share=False, server_name="0.0.0.0", server_port=7860, show_error=True, allowed_paths=["/kaggle/tmp", "/tmp", "/kaggle/working"])
    except Exception as e2:
        print(f"Second launch also failed: {e2}")
        traceback.print_exc()
        # Last resort: default launch
        demo.launch(show_error=True)
